# iRED S² and psi circular variance, all residues (PANAV, 10-seed ensemble)

Per residue over the MD trajectories. Residues: GT present, prediction mask True, no proline.

In [ ]:
import json, glob, os, time
import numpy as np
import pandas as pd
import mdtraj as mdt

RES = "../../data/md_1us"  # 1-us MD trajectories, one frame per ns
PRED = "../../data/analysis/Fig4/results/predictions_N_seed_ensemble_10seed.json"
GT_PATH = "../../data/rBMRB/cluster_30p/PANAV/test.json"
REC_PATH = "../../data/analysis/Fig4/sasa_results/per_residue_records.json"
OUT = ("../../data/analysis/Fig4/per_protein_analysis/md_1us_combined_48_justs2_justdihedral_panav10seedens.json")

ENTRIES = ["11139_1", "36329_1", "30706_1", "6032_1", "16603_1", "10267_1", "6726_1", "52187_1", "19743_1", "11202_1",
           "34106_1", "4463_1", "4819_1", "5921_1", "6933_1", "16628_1", "34043_1", "26517_1", "34425_1", "11188_1",
           "19298_1", "25895_1", "52088_1", "16387_1", "5902_1", "6782_1", "10235_1", "19737_1", "17136_1", "34607_1",
           "50270_1", "6114_1", "25968_1", "5842_1", "27687_1", "16016_1", "25448_1", "36327_1", "51126_1", "17723_1",
           "16848_1", "4671_1", "5490_1", "34678_1", "19231_1", "18145_1", "16652_1", "16186_1"]

DROP = 100

## None/Pro gate, S²_iRED / psi circular variance, JSON sanitiser

In [ ]:
def in_gate(x, r):
    """None / Proline exclusion; no 4-sigma BMRB outlier filter (already part of the prediction mask)."""
    return x is not None and r != "P"


def s2_ired_of(win, pairs, prow):
    """iRED (isotropic Reorientational Eigenmode Dynamics) N-H order parameter.

    ALIGNMENT-FREE: uses only intra-frame N-H dot products, so it needs NO superposition
    and is immune to the reference-frame problem that makes RMSF / Cartesian-S2 depend on
    which atoms define the alignment. Build the reorientational covariance
        M_ij = <P2(e_i . e_j)> = <1.5 (e_i . e_j)^2 - 0.5>   (average over frames),
    diagonalize M (diag(M) == 1 exactly), and treat the 5 largest-eigenvalue modes as
    overall isotropic tumbling. The internal order parameter is the projection kept by
    those 5 modes:
        S2_i = sum_{m in top-5} lambda_m * psi_m[i]^2   ( == 1 - internal-mode contribution ).
    Prompers & Bruschweiler, JACS 2002, 124, 4522. Returns {res_index: S2}."""
    if len(prow) < 6:                       # need > 5 vectors to remove the 5 overall modes
        return {}
    v = win.xyz[:, pairs[:, 1], :] - win.xyz[:, pairs[:, 0], :]      # [F, N, 3] N->H
    v /= np.linalg.norm(v, axis=-1, keepdims=True)
    N = v.shape[1]
    M = np.zeros((N, N))
    for f in range(v.shape[0]):             # accumulate frame-by-frame (memory safe)
        c = v[f] @ v[f].T                    # [N, N] cos(theta_ij) in frame f
        M += 1.5 * c * c - 0.5
    M /= v.shape[0]
    w, Q = np.linalg.eigh(M)                 # ascending eigenvalues, Q columns = modes
    top5 = np.argsort(w)[::-1][:5]           # 5 largest = overall tumbling
    s2 = (w[top5] * (Q[:, top5]**2)).sum(1)  # sum_m lambda_m psi_m[i]^2
    return {prow[i]: float(s2[i]) for i in range(len(prow))}


def psi_circvar_of(win):
    """Circular variance AND circular mean angle of each residue's own psi(i) dihedral,
    alignment-free (psi built from N_i, CA_i, C_i, N_{i+1} -- an internal, rotation/
    translation-invariant coordinate, same reasoning as s2_ired).

    V = 1 - |mean_f(exp(i*theta_f))|, in [0, 1]. V=0 -> psi never moves (rigid); V->1 -> psi
    samples the whole circle (disordered) -- the OPPOSITE direction from an order parameter.
    Returns (V_dict, mean_deg_dict), each {residue_index: value}, keyed by the psi's "home"
    residue i (the one owning N_i, CA_i, C_i) -- i.e. the entry for residue i is psi(i), NOT
    psi(i-1)."""
    psi_idx, psi_ang = mdt.compute_psi(win)   # idx: [n_psi,4] = (N_i, CA_i, C_i, N_{i+1})
    zbar = np.exp(1j * psi_ang).mean(axis=0)  # [n_psi] complex mean resultant vector
    V = 1.0 - np.abs(zbar)
    mean_deg = np.degrees(np.angle(zbar))
    top = win.top
    V_d    = {top.atom(psi_idx[i, 0]).residue.index: float(V[i])        for i in range(len(V))}
    mean_d = {top.atom(psi_idx[i, 0]).residue.index: float(mean_deg[i]) for i in range(len(V))}
    return V_d, mean_d


def jsonable(v):
    """numpy -> plain python, NaN -> None, so the file is strict JSON."""
    if isinstance(v, dict):
        return {k: jsonable(x) for k, x in v.items()}
    if isinstance(v, (list, tuple, np.ndarray)):
        return [jsonable(x) for x in v]
    if isinstance(v, (np.floating, float)):
        f = float(v)
        return None if not np.isfinite(f) else f
    if isinstance(v, (np.integer, int)):
        return int(v)
    if isinstance(v, (np.bool_, bool)):
        return bool(v)
    return v

## Load chemical-shift inputs

In [ ]:
ST  = json.load(open(PRED))       # seeds, pred, mask, mean, std (10-seed ensemble)
GT  = json.load(open(GT_PATH))    # experimental N, sequence -- PANAV test set (entry_ID value == pdb_id, no "bmr" prefix)
REC = json.load(open(REC_PATH))   # ss_class, exposure (from sasa_results, no s2/plddt/tdi/... fields)
key_of = {v: k for k, v in GT["entry_ID"].items()}

out = {"meta": {
    "created": time.strftime("%Y-%m-%d %H:%M:%S"),
    "trajectories": RES,
    "frame_spacing_ns": 1.0,
    "drop_frames": DROP,
    "ss_source": "per_residue_records.json (ss_class) -- no DSSP on the trajectory",
    "residue_filter": "None/Proline exclusion + prediction's own mask only -- NO persist90 (ss_ok/burial_ok) gate.",
    "alignment": "NONE for either metric: s2_ired uses N-H dot products, psi variance uses each "
                 "dihedral's own 4 atoms -- both internal, rotation/translation-invariant coordinates.",
    "s2_ired": "iRED order parameter (Prompers & Bruschweiler, JACS 2002, 124, 4522). Computed ONCE "
               "per protein over every amide N-H; None for a residue with no amide H rather than dropped.",
    "psi_var_self": "Circular variance of psi(i) = N(i)-CA(i)-C(i)-N(i+1). HIGH = MORE dynamic.",
    "psi_var_prev": "Circular variance of psi(i-1) = N(i-1)-CA(i-1)-C(i-1)-N(i), attributed to residue i.",
    "psi_var_combined": "mean(psi_var_self, psi_var_prev) over whichever exist.",
    "psi_mean_self_deg": "Circular mean angle (deg) of psi(i).",
    "psi_mean_prev_deg": "Circular mean angle (deg) of psi(i-1), attributed to residue i.",
    "acerr": "|pred - exp|, None/Proline gated. No per-protein offset correction (PANAV referencing is "
             "already consistent) and no 4-sigma BMRB outlier filter (folded into the prediction's own mask).",
    "entries": ENTRIES,
}, "proteins": {}}

## Main pass: one trajectory at a time

In [ ]:
import re, os
_FRAME_RE = re.compile(r"^frame_?(\d+)")

def _frame_idx(p):
    return int(_FRAME_RE.match(os.path.basename(p)).group(1))

def _nanmean_or_none(arr):
    arr = np.asarray(arr, float)
    return float(np.nanmean(arr)) if np.isfinite(arr).any() else None

for e in ENTRIES:
    t0 = time.time()
    gk  = key_of[e]           # PANAV entry_ID value IS the pdb_id, no "bmr" prefix
    seq = GT["sequence"][gk]
    Nv  = GT["N"][gk]
    rec = REC[e]
    L   = len(seq)

    # ---- chemical shift side (alignment-independent) ----
    pred_arr_full  = np.array(ST["mean"][e])          # 10-seed ensemble mean, not the per-seed "pred"
    pred_mask_full = np.array(ST["mask"][e], bool)
    if len(pred_arr_full) < L or len(pred_mask_full) < L:
        print(f"  !! {e}: prediction array shorter than sequence ({len(pred_arr_full)} < {L}), skipping")
        continue
    # Prediction arrays are padded to the longest sequence in the batch; any tail beyond L is
    # mask=False / mean=None padding, so truncate to L instead of requiring equal lengths
    # (the raw array is object dtype because of None; cast to float after truncation)
    pred_arr  = pred_arr_full[:L].astype(float)
    pred_mask = pred_mask_full[:L]

    gate   = np.array([in_gate(Nv[i], seq[i]) and pred_mask[i] for i in range(L)])   # None/Pro + prediction mask
    t_all  = np.array([Nv[i] if gate[i] else np.nan for i in range(L)], float)
    signed = np.where(gate, pred_arr - t_all, np.nan)
    acerr  = np.abs(signed)                     # no per-protein offset correction; NaN outside the gate

    ss       = np.array([rec[i]["ss_class"] if i < len(rec) else None for i in range(L)], dtype=object)
    exposure = np.array([rec[i]["exposure"] if i < len(rec) else None for i in range(L)], dtype=object)

    # ---- trajectory (only needed for the alignment-free s2_ired / psi variance) ----
    pdbs = sorted(
            (p for p in glob.glob(f"{RES}/{e}/frame*.pdb")
            if _FRAME_RE.match(os.path.basename(p))),
            key=_frame_idx,
        )

    win = mdt.load(pdbs)[DROP:]
    top = win.top
    nf, nres = win.n_frames, top.n_residues
    n = min(nres, L)
    if nres != L:
        print(f"  !! {e}: MD {nres} residues vs CS table {L} -- using first {n}")

    # ---- index alignment check: rec[i]["resnum"] must equal trajectory residue i (the key used by
    # the s2_ired_d / psi_var dicts). Verified for all 48 proteins that
    # rec[i]["resnum"] == i+1 == top.residue(i).resSeq; the runtime assertion guards against a future
    # data source with non-contiguous resnums or non-protein residues in the trajectory.
    for i in range(n):
        exp_resnum = i + 1
        assert rec[i]["resnum"] == exp_resnum, (
            f"{e}: per_residue_records resnum {rec[i]['resnum']} != index+1 ({exp_resnum}) at i={i} "
            f"-- s2_ired/psi_var dict keys (trajectory residue index) would misalign with rec/seq")
        assert top.residue(i).resSeq == exp_resnum, (
            f"{e}: trajectory residue {i} has resSeq {top.residue(i).resSeq} != index+1 ({exp_resnum}) "
            f"-- s2_ired/psi_var dict keys would misalign with rec/seq")

    # amide N-H pairs (for iRED S2), prow is residue index
    pairs, prow = [], []
    for r in range(n):
        res = top.residue(r)
        nat = [a.index for a in res.atoms if a.name == "N"]
        hat = [a.index for a in res.atoms if a.name in ("H", "HN", "H1")]
        if nat and hat:
            pairs.append([nat[0], hat[0]])
            prow.append(r)
    pairs = np.array(pairs) if pairs else np.zeros((0, 2), int)

    # iRED S2 (ALIGNMENT-FREE, whole molecule) -- computed ONCE, independent of any superpose()
    s2_ired_d = s2_ired_of(win, pairs, prow)

    # psi circular variance + mean (ALIGNMENT-FREE, per residue) -- straight off raw coordinates
    psi_var, psi_mean = psi_circvar_of(win)   # {residue_index: circular variance / mean angle (deg)}

    psi_self_d, psi_prev_d, psi_comb_d = {}, {}, {}
    psi_mean_self_d, psi_mean_prev_d = {}, {}
    for i in range(n):
        vals = []
        if i in psi_var:
            psi_self_d[i] = psi_var[i]
            psi_mean_self_d[i] = psi_mean[i]
            vals.append(psi_var[i])
        if (i - 1) in psi_var:
            psi_prev_d[i] = psi_var[i - 1]
            psi_mean_prev_d[i] = psi_mean[i - 1]
            vals.append(psi_var[i - 1])
        if vals:
            psi_comb_d[i] = float(np.mean(vals))

    # ---- whole-protein complete-case filter: error gate only, no persist90 restriction.
    # s2_ired / psi_var_* are NOT required to exist here -- a residue with no amide H (no
    # s2_ired) or at a chain terminus (no psi) still gets a row, just with None for that metric,
    # instead of being dropped entirely because ONE metric happened to be unavailable.
    kept = [i for i in range(n) if gate[i]]

    acerr_vals   = np.array([acerr[i] for i in kept])
    s2_ired_vals = np.array([s2_ired_d.get(i, np.nan) for i in kept], float)
    self_vals    = np.array([psi_self_d.get(i, np.nan) for i in kept], float)
    prev_vals    = np.array([psi_prev_d.get(i, np.nan) for i in kept], float)
    comb_vals    = np.array([psi_comb_d.get(i, np.nan) for i in kept], float)

    out["proteins"][e] = dict(
        bmrb_key=gk, n_frames=int(nf), n_res_md=int(nres), n_res_cs=int(L),
        n_res_used=int(n), n_complete=len(kept),
        n_res=len(kept),
        acerr=_nanmean_or_none(acerr_vals),
        s2_ired=_nanmean_or_none(s2_ired_vals),
        s2_ired_sd=(float(np.nanstd(s2_ired_vals)) if np.isfinite(s2_ired_vals).any() else None),
        psi_var_self=_nanmean_or_none(self_vals),
        psi_var_prev=_nanmean_or_none(prev_vals),
        psi_var_combined=_nanmean_or_none(comb_vals),
        per_residue=dict(
            resnum=[rec[i]["resnum"] for i in kept],
            aa=[seq[i] for i in kept],
            ss_class=[ss[i] for i in kept],
            exposure=[exposure[i] for i in kept],
            acerr=[float(acerr[i]) for i in kept],
            s2_ired=[s2_ired_d.get(i) for i in kept],
            psi_var_self=[psi_self_d.get(i) for i in kept],
            psi_var_prev=[psi_prev_d.get(i) for i in kept],
            psi_var_combined=[psi_comb_d.get(i) for i in kept],
            psi_mean_self_deg=[psi_mean_self_d.get(i) for i in kept],
            psi_mean_prev_deg=[psi_mean_prev_d.get(i) for i in kept],
        ),
    )
    p = out["proteins"][e]
    s2s   = f"{p['s2_ired']:.3f}"        if p["s2_ired"] is not None else "  n/a"
    selfs = f"{p['psi_var_self']:.3f}"   if p["psi_var_self"] is not None else "  n/a"
    prevs = f"{p['psi_var_prev']:.3f}"   if p["psi_var_prev"] is not None else "  n/a"
    aes   = f"{p['acerr']:.3f}"          if p["acerr"] is not None else "  n/a"
    print(f"{e} [{time.time()-t0:.0f}s] nf={nf} n_kept={p['n_res']} "
          f"S2iRED={s2s} psiSelf={selfs} psiPrev={prevs} acerr={aes}", flush=True)

## Write JSON

In [ ]:
print(f"proteins remaining: {len(out['proteins'])}")

with open(OUT, "w") as fh:
    json.dump(jsonable(out), fh, indent=1, allow_nan=False)
print(f"wrote {OUT}  ({os.path.getsize(OUT)/1e6:.2f} MB)")

# re-read with the strict parser as a sanity check (fails loudly if any NaN leaked)
_chk = json.load(open(OUT))
print("re-parsed OK:", len(_chk["proteins"]), "proteins")